In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/minimum_hourly_merged.csv', parse_dates=['date_local'])
df = df.sort_values('date_local').dropna(subset=['met_TMP'])

# naive forecast ("Last Observed Value")
# shift the temp column by 1 to "forecast" the current value using the previous hour
df['naive_forecast'] = df['met_TMP'].shift(1)

# linear regression
# where we use the local hour and month as predictors for temperature
df['hour_sin'] = np.sin(2 * np.pi * df['hour_local'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_local'] / 24)
df_clean = df.dropna(subset=['naive_forecast'])

X = df_clean[['hour_sin', 'hour_cos', 'month_local']]
y = df_clean['met_TMP']

# spliting the data to keep temporal order
split = int(len(df_clean) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
naive_test = df_clean['naive_forecast'].iloc[split:]

# fitting it into a linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# performance metrics base off mean squared error and R2 score
metrics = { 
    "Naive MSE": mean_squared_error(y_test, naive_test),
    "Linear Regression MSE": mean_squared_error(y_test, lr_preds),
    "Linear Regression R2": r2_score(y_test, lr_preds)
}

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Naive MSE: 2.6274
Linear Regression MSE: 31.6084
Linear Regression R2: -1.0378
